In [1]:
import os
import ssl
import httpx
import truststore
from openai import OpenAI, APIConnectionError, RateLimitError

# Reuse the same TLS strategy as Cell 1.
verify = truststore.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
verify_note = "Using truststore (Windows system certificates)."

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    http_client=httpx.Client(verify=verify, timeout=30.0),
)

try:
    response = client.responses.create(
        model="gpt-5.5",  # gpt-5.4-mini # gpt-5.4 gpt-5.5
        input="Write a one-sentence bedtime story about a unicorn."
    )
    print(response.output_text)
except RateLimitError as e:
    print("Request reached OpenAI, but the API key has no available quota (429).")
    print(f"Details: {e}")
except APIConnectionError as e:
    print("Connection failed.")
    print("If truststore is unavailable, install it with: pip install truststore")
    print("Or set SSL_CERT_FILE / REQUESTS_CA_BUNDLE to your corporate root CA PEM file.")
    print(f"Details: {e}")

Request reached OpenAI, but the API key has no available quota (429).
Details: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}


In [1]:
from skills.registry import load_skill_registry

r = load_skill_registry()
print(r.list_actions())
print(r.get_action_rule('forward'))

['backward', 'camera', 'face_to', 'forward', 'gripper_down', 'gripper_pos', 'gripper_up', 'left', 'right', 'sensor', 'stop', 'straightbackward', 'straightforward']
ActionRule(action='forward', route='move', value_type='int', skill_id='move', runtime='pc.tools.move_tools:execute', arg_key='distance_cm', allowed=(), min_value=0, max_value=10000)


In [3]:
from pc.llm.intent_mapper import intent_to_sequence
cases=[
    {'steps':[
        {'action':'forward','args':{'distance_cm':30}}]},
        {'steps':[{'action':'left','args':{'angle_deg':90}}]},
        {'steps':[{'action':'camera','args':{'mode':'photo'}}]},
        {'steps':[{'action':'sensor','args':{'name':'distance'}}]}]
[print(intent_to_sequence(case)) for case in cases]

{'sequence': [{'cmd': 'forward 30'}]}
{'sequence': [{'cmd': 'left 90'}]}
{'sequence': [{'cmd': 'camera photo'}]}
{'sequence': [{'cmd': 'sensor distance'}]}


[None, None, None, None]

In [7]:
import asyncio
from pc.agent.robot_agent import RobotAgent

class DummyHub:
    async def send(self, cmd):
        print('hub.send', cmd)
async def main():
    agent = RobotAgent(DummyHub())
    cases = [
        {'sequence':[{'cmd':'forward 30'}]},
        {'sequence':[{'cmd':'left 90'}]},
        {'sequence':[{'cmd':'camera photo'}]},
        {'sequence':[{'cmd':'sensor distance'}]},
        {'sequence':[{'cmd':'left 999'}]},
    ]
    for case in cases:
        print(await agent.execute_sequence(case))
await main()

hub.send forward 30
{'status': 'ok', 'executed': ['forward 30'], 'skipped': [], 'errors': []}
hub.send left 90
{'status': 'ok', 'executed': ['left 90'], 'skipped': [], 'errors': []}
{'status': 'ok', 'executed': ['simulated photo'], 'skipped': [], 'errors': []}
{'status': 'ok', 'executed': ['simulated distance'], 'skipped': [], 'errors': []}
Skipped command: left 999, reason: value above max
{'status': 'error', 'executed': [], 'skipped': [{'index': 0, 'cmd': 'left 999', 'reason': 'value above max'}], 'errors': []}
